In [16]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [17]:
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash'
)

In [19]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation :str

In [20]:
def Generate_joke(state: JokeState):
    prompt = f"Generate a short joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content

    return{'joke': response}

In [21]:
def Generate_explanation(state: JokeState):
    prompt = f"Generate a short explanation of the joke {state['joke']}"
    response = llm.invoke(prompt).content

    return{'explanation': response}

In [22]:
graph = StateGraph(JokeState)

graph.add_node('Generate_joke', Generate_joke)
graph.add_node('Generate_explanation', Generate_explanation)

graph.add_edge(START, 'Generate_joke')
graph.add_edge('Generate_joke', 'Generate_explanation')
graph.add_edge('Generate_explanation', END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer = checkpointer)

In [30]:
initial_state = {
    "topic": "bike",
}
config2 = {'configurable': {'thread_id': '2'}}

result = workflow.invoke(initial_state, config = config2)

In [34]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'bike', 'joke': [{'type': 'text', 'text': 'Why did the bicycle fall over?\n\nBecause it was two-tired!', 'extras': {'signature': 'Er4KCrsKARFNMg9Ak1iP/YZ3G6JlcsgQIO0bBSkCH03q9sxyAmaah4NgkQaVkcFESSQcFGTBlZUoTHOdf6HOiAD1mj0F/+zPDylYyPGg/AyVCesNgI679O2dC0SYmp4YsoEWK1Tcjh+GyRDeU0EgF0Vozz43+6KzbIvaHGdBMmTdqZfQnJluVxvwSDQP+hCwaI8wq3yY3ApVo5x2aDrGWgr7mOH96NSYxE72sZd3tivBAb5kVNAuNzdD1KegedLViIVfqT/nSSs7NaMg7kdZjD/ypwD5sIaH5GAo7+klcGYYWRoi4FAjD0K0uBcyq2Xa90iFP5hwFMm4Z4Ncz+JTR/eA1hIzyEnUuAMezu20LVzColWN9nFIJydWgR+h+E69mdCqk/4jR433mhaWKqy0Au23QLoITJzBZefMbrZE18O1j2S4ETz1XOlwk1cYjpbstQ/kppC2SPDi8FKBs9eOwYHM5QjfPqqwqpSEHwTI+iG7FxBLZvcuQRbu8lRYVaHQuqEkQpNJwmjvOevaTHwBHGJsOpeTX14isxYZPlatAhG9aocRV8jAqUbTlykOIRxeBddKx32Im3CosTifT6gMCLk/lrORVJibDosWSiOHynJeRaktHSGKKPg6ThIjPaGq0PcO8j3jVKXwjXYAz09vLjtWcwDNZPyD2z4BUnad/DHxd1orYnhN7MBfJQbpA/G3G2akukxMa2I2ZRdqRzUV4bs7E6g1Vf28LgWrbJl5AQhP560j2etX6NtzF66B9DvTRunfRUqQXEjOgQYwXonhSGRhfu0P1NF0KTc6HOD4bQ2FZIKLEXLjST2+hnfgwPuEsTH7IJT

In [32]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'bike', 'joke': [{'type': 'text', 'text': 'Why did the bicycle fall over?\n\nBecause it was two-tired!', 'extras': {'signature': 'Er4KCrsKARFNMg9Ak1iP/YZ3G6JlcsgQIO0bBSkCH03q9sxyAmaah4NgkQaVkcFESSQcFGTBlZUoTHOdf6HOiAD1mj0F/+zPDylYyPGg/AyVCesNgI679O2dC0SYmp4YsoEWK1Tcjh+GyRDeU0EgF0Vozz43+6KzbIvaHGdBMmTdqZfQnJluVxvwSDQP+hCwaI8wq3yY3ApVo5x2aDrGWgr7mOH96NSYxE72sZd3tivBAb5kVNAuNzdD1KegedLViIVfqT/nSSs7NaMg7kdZjD/ypwD5sIaH5GAo7+klcGYYWRoi4FAjD0K0uBcyq2Xa90iFP5hwFMm4Z4Ncz+JTR/eA1hIzyEnUuAMezu20LVzColWN9nFIJydWgR+h+E69mdCqk/4jR433mhaWKqy0Au23QLoITJzBZefMbrZE18O1j2S4ETz1XOlwk1cYjpbstQ/kppC2SPDi8FKBs9eOwYHM5QjfPqqwqpSEHwTI+iG7FxBLZvcuQRbu8lRYVaHQuqEkQpNJwmjvOevaTHwBHGJsOpeTX14isxYZPlatAhG9aocRV8jAqUbTlykOIRxeBddKx32Im3CosTifT6gMCLk/lrORVJibDosWSiOHynJeRaktHSGKKPg6ThIjPaGq0PcO8j3jVKXwjXYAz09vLjtWcwDNZPyD2z4BUnad/DHxd1orYnhN7MBfJQbpA/G3G2akukxMa2I2ZRdqRzUV4bs7E6g1Vf28LgWrbJl5AQhP560j2etX6NtzF66B9DvTRunfRUqQXEjOgQYwXonhSGRhfu0P1NF0KTc6HOD4bQ2FZIKLEXLjST2+hnfgwPuEsTH7IJ

In [ ]:
#here both config1 and config2 are stored in the memory, we can see them usning their thread_id.